# Swiss Commutes: audit and fixes

Reads the preserved 16-location audit offline. Counts are workers, not vehicles; city rows overlap. Geneva covers a canton and other pages municipalities. Passing arithmetic checks does not establish actual mode choices, traffic, attendance or route use.

In [1]:
from pathlib import Path
import json
root = Path.cwd()
if not (root / 'audit.json').exists(): root = root / 'audits/2026-09-06-fixed'
audit = json.loads((root / 'audit.json').read_text())
findings = json.loads((root / 'findings.json').read_text())
failures = [r for r in audit['checks'] if r['status']=='fail']
print('Snapshot:', audit['generatedAt'])
print('Locations:', len(audit['cityRows']), 'Corridors:', len(audit['routes']))
print('Checks:', len(audit['checks']), 'Failures:', len(failures))
for r in failures: print(r)


Snapshot: 2026-09-06T19:14:15.057Z
Locations: 16 Corridors: 25169
Checks: 642 Failures: 0


## Disposition of the original findings

Open and partly corrected findings remain material even when numerical checks pass. The neighbouring 2026-09-06 directory preserves the baseline.

In [2]:
for r in findings: print(r['id'], '|', r['status'], '|', r['title'])

F01 | Partly corrected | Direction-specific mode shares
F02 | Corrected | Shared journeys agree across pages
F03 | Corrected | Motorcycles are disclosed
F04 | Corrected | No-journey records excluded
F05 | Open limitation | Active-mode residual
F06 | Corrected | Geneva uses a consistent Swiss scope
F07 | Corrected | Geneva workplaces retained
F08 | Partly corrected | Coverage disclosed
F09 | Partly corrected | Unrouted active estimates removed from chart
F10 | Corrected | One journey clock
F11 | Open limitation | Illustrative working day
F12 | Open limitation | Rail paths are possible geometry
F13 | Open limitation | Foreign origins outside Geneva
F14 | Partly corrected | Vintages and exclusions exposed
F15 | Corrected | Road endpoint review
F16 | Partly corrected | Reproducible inputs and checks


## Reciprocal journeys and timing

Compare shared Swiss municipal pairs across pages and independent arrival accounting on the same routed car cohort. Geneva has a different geography and vintage and is excluded from reciprocal municipal comparisons.

In [3]:
assert all(r['delta']==0 for r in audit['reciprocal'])
assert all(r['populationCarDisagreement']==0 for r in audit['cityRows'])
assert audit['french']['mismatches']==0 and not audit['french']['omitted']
print('Shared OD comparisons:', len(audit['reciprocal']))
print('French workplace/mode rows:', audit['french']['comparedRows'])
print('Excluded no-journey survey weight:', audit['french']['byOriginalMode']['1'])


Shared OD comparisons: 128
French workplace/mode rows: 3130
Excluded no-journey survey weight: 157.64459489058729


## Routing coverage

The denominator is the mapped estimate, after geographic selection. Coverage is completeness, not accuracy. Long-distance active estimates remain in the evidence but are excluded from the chart.

In [4]:
for city in audit['cityRows']:
    parts=[]
    for mode in ['car','transit','soft']:
        rows=[r for r in audit['modes'] if r['city']==city['city'] and r['mode']==mode]
        mapped=sum(r['people'] for r in rows); routed=sum(r['covered'] for r in rows)
        parts.append(f'{mode}: {routed:,}/{mapped:,} ({routed/mapped:.1%})' if mapped else f'{mode}: no estimates')
    print(city['name'] + ' | ' + ' | '.join(parts))


Zürich | car: 82,348/86,144 (95.6%) | transit: 162,472/203,572 (79.8%) | soft: 5,430/8,112 (66.9%)
Geneva | car: 98,985/104,193 (95.0%) | transit: 9,210/28,455 (32.4%) | soft: 8,518/13,067 (65.2%)
Basel | car: 29,743/30,953 (96.1%) | transit: 28,266/63,047 (44.8%) | soft: 21,822/24,445 (89.3%)
Lausanne | car: 34,814/36,625 (95.1%) | transit: 22,246/36,333 (61.2%) | soft: 13,535/20,787 (65.1%)
Bern | car: 38,785/40,577 (95.6%) | transit: 53,611/69,146 (77.5%) | soft: 5,287/7,951 (66.5%)
Winterthur | car: 26,464/27,758 (95.3%) | transit: 30,051/35,859 (83.8%) | soft: 1,926/2,600 (74.1%)
Lucerne | car: 24,444/25,566 (95.6%) | transit: 22,625/27,681 (81.7%) | soft: 3,734/4,894 (76.3%)
St. Gallen | car: 31,637/33,256 (95.1%) | transit: 18,823/25,194 (74.7%) | soft: 954/1,620 (58.9%)
Lugano | car: 20,195/21,245 (95.1%) | transit: 2,906/9,825 (29.6%) | soft: 7,729/9,889 (78.2%)
Biel/Bienne | car: 13,292/13,961 (95.2%) | transit: 6,702/9,834 (68.2%) | soft: 5,525/7,455 (74.1%)
Schaffhausen | c

## Largest missing routes

These are missing geometries for estimated categories, not confirmed missing train services. Filter the complete corridors.csv for other views; retain the category, scope and source limitations.

In [5]:
missing=sorted((r for r in audit['routes'] if r['geometry']=='missing'),key=lambda r:r['commuters'],reverse=True)
for r in missing[:15]: print(r['city'], r['originName'], '→', r['targetName'], r['mode'], r['commuters'])
assert not failures, 'Review failed checks before treating the build as numerically reconciled'


basel Saint-Louis → Basel transit 2746
basel Binningen → Basel transit 2242
basel Reinach (BL) → Basel transit 1805
basel Huningue → Basel transit 1685
zurich Volketswil → Zürich transit 1574
basel Grenzach-Wyhlen → Basel transit 1496
basel Birsfelden → Basel transit 1420
zurich Fällanden → Zürich transit 1344
basel Oberwil (BL) → Basel transit 1262
bern Wohlen bei Bern → Bern transit 1236
zurich Oberengstringen → Zürich transit 1184
basel Basel → Allschwil transit 1100
geneva Gaillard → Genève transit 1090
basel Therwil → Basel transit 1027
zurich Herrliberg → Zürich transit 1007
